In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from xgboost import XGBClassifier



In [72]:
df = pd.read_csv('data/data.csv')


In [73]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.1 MB


In [74]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Drop columns
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

# Split
y = df['Exited']
X = df.drop(columns=['Exited'])

# Column groups
scale_cols = ['CreditScore', 'Age', 'Balance', 'EstimatedSalary']
cat_cols = X.select_dtypes(include='object').columns
pass_cols = [col for col in X.columns if col not in scale_cols and col not in cat_cols]

# Transformer (IMPORTANT CHANGE HERE)
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), scale_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
], remainder='passthrough')

# Transform (no toarray needed now)
data = preprocessor.fit_transform(X)

# Column names
cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols)
all_cols = list(scale_cols) + list(cat_features) + pass_cols

# Final DataFrame
final_df = pd.DataFrame(data, columns=all_cols)
final_df['Exited'] = y.values

print(final_df.head())

   CreditScore       Age   Balance  EstimatedSalary  Geography_France  \
0    -0.326221  0.293517 -1.225848         0.021886               1.0   
1    -0.440036  0.198164  0.117350         0.216534               0.0   
2    -1.536794  0.293517  1.333053         0.240687               1.0   
3     0.501521  0.007457 -1.225848        -0.108918               1.0   
4     2.063884  0.388871  0.785728        -0.365276               0.0   

   Geography_Germany  Geography_Spain  Gender_Female  Gender_Male  Tenure  \
0                0.0              0.0            1.0          0.0     2.0   
1                0.0              1.0            1.0          0.0     1.0   
2                0.0              0.0            1.0          0.0     8.0   
3                0.0              0.0            1.0          0.0     1.0   
4                0.0              1.0            1.0          0.0     2.0   

   NumOfProducts  HasCrCard  IsActiveMember  Exited  
0            1.0        1.0             1.0 

In [75]:
final_df.head()

,CreditScore,Age,Balance,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain,Gender_Female,Gender_Male,Tenure,NumOfProducts,HasCrCard,IsActiveMember,Exited
0,-0.326221,0.293517,-1.225848,0.021886,1.0,0.0,0.0,1.0,0.0,2.0,1.0,1.0,1.0,1
1,-0.440036,0.198164,0.117350,0.216534,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0
2,-1.536794,0.293517,1.333053,0.240687,1.0,0.0,0.0,1.0,0.0,8.0,3.0,1.0,0.0,1
3,0.501521,0.007457,-1.225848,-0.108918,1.0,0.0,0.0,1.0,0.0,1.0,2.0,0.0,0.0,0
4,2.063884,0.388871,0.785728,-0.365276,0.0,0.0,1.0,1.0,0.0,2.0,1.0,1.0,1.0,0


In [79]:
y = final_df['Exited']
X = final_df.drop(columns=['Exited'])

In [80]:
X

,CreditScore,Age,Balance,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain,Gender_Female,Gender_Male,Tenure,NumOfProducts,HasCrCard,IsActiveMember
0,-0.326221,0.293517,-1.225848,0.021886,1.0,0.0,0.0,1.0,0.0,2.0,1.0,1.0,1.0
1,-0.440036,0.198164,0.117350,0.216534,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0
2,-1.536794,0.293517,1.333053,0.240687,1.0,0.0,0.0,1.0,0.0,8.0,3.0,1.0,0.0
3,0.501521,0.007457,-1.225848,-0.108918,1.0,0.0,0.0,1.0,0.0,1.0,2.0,0.0,0.0
4,2.063884,0.388871,0.785728,-0.365276,0.0,0.0,1.0,1.0,0.0,2.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1.246488,0.007457,-1.225848,-0.066419,1.0,0.0,0.0,0.0,1.0,5.0,2.0,1.0,0.0
9996,-1.391939,-0.373958,-0.306379,0.027988,1.0,0.0,0.0,0.0,1.0,10.0,1.0,1.0,1.0
9997,0.604988,-0.278604,-1.225848,-1.008643,1.0,0.0,0.0,1.0,0.0,7.0,1.0,0.0,1.0
9998,1.256835,0.293517,-0.022608,-0.125231,0.0,1.0,0.0,0.0,1.0,3.0,2.0,1.0,0.0
